In [1]:
# Imports
from dotenv import load_dotenv
from anthropic import Anthropic
from building_with_the_claude_api import add_user_message, add_assistant_message, chat, Effort
from building_with_the_claude_api.prompt_evaluator import PromptEvaluator, generate_prompt_evaluation_report


In [2]:
# Client Initialization and helper functions

load_dotenv()

client = Anthropic()

model = "claude-haiku-4-5"
# model = "claude-sonnet-4-6"


In [3]:
# Create an instance of PromptEvaluator
# Increase `max_concurrent_tasks` for greater concurrency, but beware of rate limit errors!
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [4]:
dataset = evaluator.generate_dataset(
    client=client,
    model=model,
    # Describe the purpose or goal of the prompt you're trying to test
    task_description="write a compact, concise 1 day meal plan for a single athlete",
    # Describe the different inputs that your prompt requires
    prompt_inputs_spec={
        "height": "Athlete height in cm",
        "weight": "Athlete weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete",
    },
    # Number of test cases to generate (recommend keeping this low if you're getting rate limit errors)
    num_cases=3,
)

Generated 1/3 test cases
Generated 2/3 test cases
Generated 3/3 test cases


In [5]:
# Define and run the prompt you want to evaluate, returning the raw model output
# This function is executed once for each test case
def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a one-day meal plan for an athlete that meets their dietary restrictions.

    <athlete_information>
    - Height: {prompt_inputs["height"]}
    - weight: {prompt_inputs["weight"]}
    - Goal: {prompt_inputs["goal"]}
    - Dietary restrictions:  {prompt_inputs["restrictions"]}
    </athlete_information>

    Use either of the following guidelines or steps to respond.
    Guidelines (variant 1 being specific):
    1. Include accurate daily calorie amount
    2. Show protein, fat, and carb amounts
    3. Specify when to eat each meal
    4. Use only foods that fit restrictions
    5. List all portion sizes in grams
    6. Keep budget-friendly if mentioned

    Follow these steps (variant 2 providing steps):
    1. Calculate daily calories needed
    2. Figure out protein, fat, carb amounts
    3. Plan meal timing around workouts
    4. Choose foods that fit restrictions
    5. Set portion sizes in grams
    6. Adjust for budget if needed

    Here is an example with a sample input and an ideal output:
    <sample_input>
        height: 178
        weight: 68
        goal: Prepare for marathon in 2 weeks with sustained energy and adequate plant-based protein
        restrictions: Vegan diet, no animal products
    </sample_input>

    <good_but_improvable_output>
        # One-Day Marathon Preparation Meal Plan for Vegan Athlete

        ## Daily Nutritional Targets
        - **Daily Calories:** 2,850 kcal
        - **Protein:** 114g (16%)
        - **Carbohydrates:** 427g (60%)
        - **Fat:** 79g (25%)

        *Based on 68kg body weight, high endurance activity level, and 2-week marathon preparation goal. Plant-based protein sources optimized for sustained energy and recovery.*

        ---

        ## MEAL SCHEDULE

        ### **BREAKFAST** (7:00 AM)
        *Calories: 580 | Protein: 22g | Carbs: 86g | Fat: 16g*

        - Oatmeal (rolled oats): 80g dry
        - Banana: 150g
        - Almond butter: 30g
        - Chia seeds: 15g
        - Plant-based milk (unsweetened): 240ml
        - Maple syrup: 15g

        **Prep:** Cook oatmeal with plant-based milk, top with sliced banana, almond butter, chia seeds, and maple syrup.

        ---

        ### **MID-MORNING SNACK** (10:00 AM)
        *Calories: 320 | Protein: 18g | Carbs: 48g | Fat: 8g*

        - Hummus: 100g
        - Whole grain pita bread: 2 pieces (80g)
        - Carrot sticks: 100g
        - Dates: 40g

        **Prep:** No cooking required. Serve at room temperature for easy consumption.

        ---

        ### **LUNCH** (1:00 PM)
        *Calories: 720 | Protein: 26g | Carbs: 108g | Fat: 18g*

        - Cooked lentils (red): 200g
        - Quinoa (cooked): 150g
        - Roasted bell peppers: 120g
        - Spinach (raw): 80g
        - Olive oil: 12ml
        - Lemon juice & garlic: to taste
        - Salt & pepper: to taste

        **Prep:** Cook lentils and quinoa separately, roast peppers, combine all ingredients with olive oil and lemon juice for a complete plant-based protein bowl.

        ---

        ### **PRE-TRAINING SNACK** (3:30 PM)
        *Calories: 350 | Protein: 16g | Carbs: 58g | Fat: 8g*

        - Tofu (firm, baked): 150g
        - Brown rice cakes: 3 pieces (45g)
        - Tahini: 20g
        - Apple: 180g
        - Water: 500ml

        **Timing:** 90 minutes before evening training run for optimal digestion and sustained energy release.

        ---

        ### **DINNER** (7:30 PM)
        *Calories: 580 | Protein: 28g | Carbs: 82g | Fat: 15g*

        - Chickpea pasta (whole grain): 180g cooked
        - Marinara sauce (no oil added): 200g
        - Nutritional yeast: 20g
        - Steamed broccoli: 150g
        - Extra virgin olive oil: 10ml
        - Fresh basil & oregano: to taste

        **Prep:** Boil pasta, warm sauce, steam broccoli, toss with olive oil and nutritional yeast for complete amino acid profile.

        ---

        ### **EVENING RECOVERY SNACK** (9:30 PM)
        *Calories: 320 | Protein: 20g | Carbs: 45g | Fat: 8g*

        - Vegan protein powder (pea/hemp blend): 30g (1 scoop)
        - Oat milk: 240ml
        - Peanut butter: 20g
        - Blueberries (frozen): 100g

        **Prep:** Blend all ingredients until smooth. Aids muscle recovery and repair overnight.

        ---

        ## HYDRATION PLAN
        - **Morning to Lunch:** 1.5L water
        - **Lunch to Pre-Training Snack:** 750ml water
        - **Training to Dinner:** 1L water + 500ml homemade sports drink (4% carbs: coconut water + dates)
        -
    </good_but_improvable_output>

    <reasoning>
        The solution comprehensively meets all mandatory requirements: it includes daily caloric total (2,850 kcal), complete macronutrient breakdown with percentages, all 6 meals with exact foods, precise portions in grams/ml, and specific timing throughout the day. It exceeds secondary criteria by incorporating far more than 3 plant-based protein sources and providing well-designed sustained carbohydrate loading appropriate for marathon training. The format is compact and well-organized. The minor weakness is the incomplete hydration plan section at the end, which doesn't affect the core meal plan but represents incomplete communication. All foods are verifiably vegan (no animal products present).
    </reasoning>
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages=messages, client=client, model=model, effort=Effort.LOW)


In [6]:
results = evaluator.run_evaluation(
    run_prompt_function=run_prompt, client=client, model=model, extra_criteria = """
    The output should include:
    - Daily caloric total
    - Macronutrient breakdown
    - Meals with exact foods, portions and timing
    """
)

Graded 1/3 test cases
Graded 2/3 test cases
Graded 3/3 test cases
Average score: 5
